# Structural and functional brain graph fusion

Five seeds each generate 2,500 training, 600 validation and 900 held-out pairs of 24-node graphs. One graph records simulated connection counts and the other comes from correlations between regional time series. Five feature sets are compared. Validation Brier score selects regularization separately for each. The stress test perturbs count rates and signal noise.

Published result: On shifted graphs, raw edge fusion reached 0.745 AUROC. Pooling by module reached 0.812, and adding cross-modal interactions reached 0.813. The simulator supplies the module labels and embeds signal at that scale. The result measures the value of that prior in this setting.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'multimodal-brain-graph-heavy-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", 'numpy==2.2.6', 'scikit-learn==1.7.2'], check=True)

### train.py

In [ ]:
(root / 'train.py').write_text('"""Module-aware structural and functional edge fusion on simulated paired graphs."""\n\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import roc_auc_score, brier_score_loss\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nN = 24\nI, J = np.triu_indices(N, 1)\nMODULE = np.arange(N) // 6\nBLOCKS = [(MODULE[I] == a) & (MODULE[J] == b) for a in range(4) for b in range(a, 4)]\n\n\ndef generate(n, seed, shift):\n    rng = np.random.default_rng(seed)\n    latent = rng.normal(size=(n, 2))\n    labels = (\n        latent[:, 0] + latent[:, 1] + 0.8 * latent.prod(1) + rng.normal(0, 0.65, n) > 0\n    )\n    struct = []\n    func = []\n    for u, v in latent:\n        rate = np.full(len(I), 0.9) + 0.85 * (MODULE[I] == MODULE[J])\n        rate[BLOCKS[0]] += 0.40 * np.tanh(u)\n        a = rng.poisson(rate * np.exp(rng.normal(0, 0.14) + 0.18 * shift))\n        shared = rng.normal(size=(72, 4))\n        load = np.full(N, 0.65)\n        load[MODULE == 2] += 0.33 * np.tanh(v)\n        ts = shared[:, MODULE] * load + rng.normal(size=(72, N)) * (1 + 0.18 * shift)\n        struct.append(np.log1p(a))\n        func.append(np.arctanh(np.clip(np.corrcoef(ts.T)[I, J], -0.995, 0.995)))\n    return np.array(struct), np.array(func), labels\n\n\ndef features(s, f):\n    def pool(x):\n        return np.stack([x[:, b].mean(1) for b in BLOCKS], 1)\n\n    a, b = pool(s), pool(f)\n    interactions = (a[:, :, None] * b[:, None, :]).reshape(len(a), -1)\n    return {\n        "structural": s,\n        "functional": f,\n        "edge_concat": np.c_[s, f],\n        "pooled": np.c_[a, b],\n        "pooled_interactions": np.c_[a, b, interactions],\n    }\n\n\ndef run(seed):\n    splits = [\n        generate(n, seed + offset, shift)\n        for n, offset, shift in [(2500, 0, 0), (600, 1, 0), (900, 2, 0), (900, 2, 1)]\n    ]\n    inputs = [features(s, f) for s, f, _ in splits]\n    result = {}\n    for method in inputs[0]:\n        candidates = []\n        for c in [0.01, 0.1, 1.0]:\n            model = make_pipeline(\n                StandardScaler(), LogisticRegression(C=c, max_iter=400)\n            )\n            model.fit(inputs[0][method], splits[0][2])\n            v = model.predict_proba(inputs[1][method])[:, 1]\n            candidates.append((brier_score_loss(splits[1][2], v), model, c))\n        _, model, c = min(candidates, key=lambda t: t[0])\n        dest = Path(__file__).with_name("checkpoints")\n        dest.mkdir(exist_ok=True)\n        scaler, lr = model.steps[0][1], model.steps[1][1]\n        np.savez_compressed(\n            dest / f"{method}-{seed}.npz",\n            mean=scaler.mean_,\n            scale=scaler.scale_,\n            coef=lr.coef_,\n            intercept=lr.intercept_,\n        )\n        for k, tag in [(2, "clean"), (3, "shifted")]:\n            p = model.predict_proba(inputs[k][method])[:, 1]\n            result[method + "_" + tag] = {\n                "auroc": roc_auc_score(splits[k][2], p),\n                "brier": brier_score_loss(splits[k][2], p),\n                "C": c,\n            }\n    return result\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in [17, 29, 43, 59, 71]}\n    means = {\n        k: {m: np.mean([r[k][m] for r in runs.values()]) for m in ["auroc", "brier"]}\n        for k in next(iter(runs.values()))\n    }\n    Path(__file__).with_name("results.json").write_text(\n        json.dumps({"runs": runs, "means": means}, indent=2)\n    )\n', encoding="utf8")

### Train and evaluate

In [ ]:
subprocess.run([sys.executable, 'train.py'], cwd=root, check=True)

In [ ]:
import json
report = json.loads((root / "results.json").read_text())
print(json.dumps(report.get("means", report.get("subject_macro")), indent=2))